# TODO

In [1]:
ENCODERS = [
    "all-mpnet-base-v2",
]

ENCODE_BATCH_SIZE = 32

AE_EPOCH_N = 100
AE_BATCH_SIZE = 8192

device = "mps"

In [2]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, auc
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import re
import random
from tqdm import tqdm
from itertools import product
from collections import defaultdict
import copy

import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/artfultom/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [5]:
filenames = [
    "../datasets/mistral_essays_1.csv",
    "../datasets/mistral_essays_2.csv",
    "../datasets/mistral_essays_3.csv",
    "../datasets/mistral_essays_4.csv",
    "../datasets/mistral_essays_5.csv",
    "../datasets/llama_essays_1.csv",
    "../datasets/llama_essays_2.csv",
    "../datasets/llama_essays_3.csv",
    "../datasets/llama_essays_4.csv",
    "../datasets/llama_essays_5.csv",
    "../datasets/deepseek_essays_1.csv",
    "../datasets/deepseek_essays_2.csv",
    "../datasets/deepseek_essays_3.csv",
    "../datasets/deepseek_essays_4.csv",
    "../datasets/deepseek_essays_5.csv",
    "../datasets/chatgpt_essays_1.csv",
    "../datasets/chatgpt_essays_2.csv",
    "../datasets/chatgpt_essays_3.csv",
    "../datasets/chatgpt_essays_4.csv",
    "../datasets/chatgpt_essays_5.csv",
]
df_ai = pd.concat([pd.read_csv(f) for f in filenames], ignore_index=True).assign(label=1)[["text", "label"]]#.sample(n=50, random_state=SEED)

df_human = pd.read_csv("../datasets/ivy_panda_essays.csv").assign(label=0)[["text", "label"]].reset_index(drop=True)#.sample(n=100, random_state=SEED)
df_train, df_human_test = train_test_split(df_human, test_size=len(df_ai), random_state=SEED)

df_all = pd.concat([df_ai, df_human_test], ignore_index=True).sample(frac=1, random_state=SEED)

In [6]:
print(f"Загружено {len(df_train)} записей Ivy Panda (живые, train)")
print(f"Загружено {len(df_all)} записей (синтетика + живые, test)")

Загружено 90805 записей Ivy Panda (живые, train)
Загружено 74976 записей (синтетика + живые, test)


In [7]:
def print_big_header(text, width=100):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)

In [8]:
def normalize_f(data):
    data = torch.from_numpy(data).float()
    data = F.normalize(data, p=2, dim=0)
    return data.numpy()

def get_raw_embeddings(model, data, normalize_embeddings=False):
    all_sentences = []
    spans = []

    for text in tqdm(data):
        text = re.sub(r"\n+", ". ", text)
        sentences = nltk.sent_tokenize(text)

        start = len(all_sentences)
        all_sentences.extend(sentences)
        end = len(all_sentences)

        spans.append((start, end))

    all_embeddings = model.encode(
        all_sentences,
        batch_size=ENCODE_BATCH_SIZE,
        normalize_embeddings=normalize_embeddings,
        show_progress_bar=True,
        device=device,
    )

    return spans, all_embeddings

def get_embeddings(spans, all_embeddings):
    mean_vals = []

    max_vals = []
    mean_diff_vals = []
    med_diff_vals = []
    var_diff_vals = []

    for start, end in tqdm(spans):
        embeddings = all_embeddings[start:end]

        mean_val = np.mean(embeddings, axis=0)

        max_val = np.max(embeddings, axis=0)
        max_val = normalize_f(max_val)

        diffs = embeddings[1:] - embeddings[:-1]
        
        mean_diff_val = np.mean(diffs, axis=0)
        mean_diff_val = normalize_f(mean_diff_val)

        med_diff_val = np.median(diffs, axis=0)
        med_diff_val = normalize_f(med_diff_val)

        var_diff_val = np.var(diffs, axis=0)
        var_diff_val = normalize_f(var_diff_val)

        mean_vals.append(mean_val)
        max_vals.append(max_val)
        mean_diff_vals.append(mean_diff_val)
        med_diff_vals.append(med_diff_val)
        var_diff_vals.append(var_diff_val)

    return (
        mean_vals,

        max_vals,
        mean_diff_vals,
        med_diff_vals,
        var_diff_vals,
    )

In [9]:
def load_or_compute_embeddings(model_name, df_train, df_all, cache_dir="../emb_cache"):
    os.makedirs(cache_dir, exist_ok=True)

    safe_model_name = model_name.replace("/", "__")

    train_path = os.path.join(cache_dir, f"{safe_model_name}_train.npz")
    test_path = os.path.join(cache_dir, f"{safe_model_name}_test.npz")

    if os.path.exists(train_path) and os.path.exists(test_path):
        print(f"[CACHE] Loading embeddings for {model_name}")

        train_data = np.load(train_path, allow_pickle=True)
        test_data = np.load(test_path, allow_pickle=True)

        train_embeddings = [
            np.asarray(e, dtype=np.float32)
            for e in train_data["embeddings"]
        ]
        test_embeddings = [
            np.asarray(e, dtype=np.float32)
            for e in test_data["embeddings"]
        ]

        return train_embeddings, test_embeddings

    print(f"[COMPUTE] Encoding {model_name}")
    model = SentenceTransformer(model_name)

    spans_train, all_embeddings_train = get_raw_embeddings(
        model,
        df_train['text'].tolist(),
        True
    )
    spans_test, all_embeddings_test = get_raw_embeddings(
        model,
        df_all['text'].tolist(),
        True
    )

    train_embeddings = list(get_embeddings(spans_train, all_embeddings_train))
    test_embeddings = list(get_embeddings(spans_test, all_embeddings_test))

    np.savez_compressed(
        train_path,
        embeddings=np.array(train_embeddings, dtype=object)
    )
    np.savez_compressed(
        test_path,
        embeddings=np.array(test_embeddings, dtype=object)
    )

    print(f"[SAVED] {model_name}")

    return train_embeddings, test_embeddings

## Автоэнкодеры
### Vanilla Autoencoder (классический)

Аномалия = высокая reconstruction error.

Особенности:
- Детеминированный латентный код.
- Только reconstruction loss (MSE).

Плюсы:
- Простота реализации.
- Быстро обучается.

Минусы:
- Склонен к переобучению
- Может выучить тождественное отображение
- Латентное пространство неструктурировано

### Denoising Autoencoder (DAE)

Учится восстанавливать чистый вход из зашумленного.

Особенности:
- Добавляется Gaussian noise.
- Учит устойчивые признаки.

Плюсы:
- Более робастные представления.
- Лучше обобщает.
- Меньше переобучается.

Минусы:
- Чувствителен к уровню шума

### Variational Autoencoder (VAE)

Вероятностный автоэнкодер с регуляризацией распределения латента.

Особенности:
- Добавляет KL-дивергенцию.

Плюсы:
- Непрерывное структурированное латентное пространство.
- Теоретически более корректная модель распределения.

Минусы:
- Может давать низкий loss на OOD. Эксперименты ниже это покажут.
- Возможен posterior collapse - игнорирование латентного пространства.

### Contractive Autoencoder (CAE)

Reconstruction error, но латент стабилен локально.

Плюсы:
- Хорошо аппроксимирует manifold нормальных данных.
- OOD вне manifold - сильный рост ошибки.
- Теоретически хорошо подходит для anomaly detection.

### Sparse Autoencoder

Плюсы:
- Нормальные данные занимают компактную область.

Минусы:
- Если эмбеддинги уже плотные - работает хуже.
- Появляются дополнительные параметры.

In [10]:
# Vanilla Autoencoder (AE)
class AE(nn.Module):
    def __init__(self, d, params):
        super().__init__()
        
        input_dim = d
        latent_dim = params['latent_dim']
        self.enc = nn.Sequential(
            nn.Linear(input_dim, latent_dim * 2),
            nn.ReLU(),
            nn.Linear(latent_dim * 2, latent_dim),
        )
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2),
            nn.ReLU(),
            nn.Linear(latent_dim * 2, input_dim),
        )
        
    def forward(self, x):
        return self.dec(self.enc(x))
        
    def compute_loss(self, x, params):
        recon = self.forward(x)
        return F.mse_loss(recon, x, reduction='mean')

# Denoising Autoencoder (DAE)
class DenoisingAE(nn.Module):
    def __init__(self, d, params):
        super().__init__()
        
        input_dim = d
        latent_dim = params['latent_dim']
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, latent_dim * 2),
            nn.ReLU(),
            nn.Linear(latent_dim * 2, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2),
            nn.ReLU(),
            nn.Linear(latent_dim * 2, input_dim)
        )
        self.noise_std = params['noise_std']

    def forward(self, x):
        if self.training:
            noise = torch.randn_like(x) * self.noise_std
            x = x + noise

        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

    def compute_loss(self, x, params):
        recon = self.forward(x)
        return F.mse_loss(recon, x, reduction='mean')

# Variational Autoencoder (VAE)
class VAE(nn.Module):
    def __init__(self, d, params):
        super().__init__()
        
        input_dim = d
        latent_dim = params['latent_dim']
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, latent_dim * 2),
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(latent_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(latent_dim * 2, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2),
            nn.ReLU(),
            nn.Linear(latent_dim * 2, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar

    def compute_loss(self, x, params):
        x_hat, mu, logvar = self.forward(x)

        recon_loss = F.mse_loss(x_hat, x, reduction='mean')

        kl = -0.5 * torch.sum(
            1 + logvar - mu.pow(2) - logvar.exp(),
            dim=1
        ).mean()

        beta = params['beta']
        return recon_loss + beta * kl

# Contractive Autoencoder (CAE)
class ContractiveAE(nn.Module):
    def __init__(self, d, params):
        super().__init__()
        
        input_dim = d
        latent_dim = params['latent_dim']
        self.encoder = nn.Linear(input_dim, latent_dim)
        self.decoder = nn.Linear(latent_dim, input_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        z = self.relu(self.encoder(x))
        x_hat = self.decoder(z)
        return x_hat, z

    def compute_loss(self, x, params):
        x_hat, h = self.forward(x)
        mse = F.mse_loss(x_hat, x, reduction='mean')

        W = self.encoder.weight
        W_norm = W.pow(2).sum(dim=1)

        dh = (h > 0).float()

        contractive = torch.mean(torch.sum(dh * W_norm, dim=1))

        lam = params['lam']
        return mse + lam * contractive

# Sparse Autoencoder (SAE)
class SparseAE(nn.Module):
    def __init__(self, d, params):
        super().__init__()
        
        input_dim = d
        latent_dim = params['latent_dim']
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, latent_dim * 2),
            nn.ReLU(),
            nn.Linear(latent_dim * 2, latent_dim),
            nn.Sigmoid()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 2),
            nn.ReLU(),
            nn.Linear(latent_dim * 2, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

    def compute_loss(self, x, params):
        rho = params['rho']
        beta = params['beta']
        
        x_hat, z = self.forward(x)

        mse = F.mse_loss(x_hat, x, reduction='mean')

        eps = 1e-8
        rho_hat = torch.mean(z, dim=0).clamp(eps, 1 - eps)
        kl = rho * torch.log(rho / rho_hat) + (1 - rho) * torch.log((1 - rho) / (1 - rho_hat))
    
        return mse + beta * kl.sum()

In [11]:
def ae_score(
    X_train,
    X_test,
    params,
    patience=10,
    min_delta=1e-5,
    val_split=0.1,
    max_epochs=AE_EPOCH_N,
):
    AE_model_class = params.get('AE_model_class')
    lr = params.get('lr')
    weight_decay = params.get('weight_decay')
    batch_size = params.get('batch_size')
    latent_dim = params.get('latent_dim')
    d = X_train.shape[1]

    n = len(X_train)
    idx = np.random.permutation(n)
    split = int(n * (1 - val_split))

    train_idx = idx[:split]
    val_idx = idx[split:]

    Xtr = torch.tensor(X_train[train_idx], dtype=torch.float32)
    Xval = torch.tensor(X_train[val_idx], dtype=torch.float32)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xval),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    ae = AE_model_class(d, params).to(device)
    
    opt = torch.optim.AdamW(ae.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=max_epochs,
        eta_min=lr * 1e-3,
    )

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(max_epochs):
        ae.train()
        for (batch,) in train_loader:
            batch = batch.to(device)

            loss = ae.compute_loss(batch, params)

            opt.zero_grad()
            loss.backward()
            opt.step()

        scheduler.step()
        ae.eval()
        val_loss = 0.0
        n_val = 0

        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(device)
                
                loss = ae.compute_loss(batch, params)

                val_loss += loss.item() * batch.size(0)
                n_val += batch.size(0)

        val_loss /= n_val

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_state = copy.deepcopy(ae.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break
    
    if best_state is not None:
        ae.load_state_dict(best_state)

    ae.eval()
    Xe = torch.tensor(X_test, dtype=torch.float32).to(device)

    with torch.no_grad():
        output = ae(Xe)
        if isinstance(output, tuple):
            recon = output[0]
        else:
            recon = output
    
        recon = recon.cpu().numpy()

    return np.mean((X_test - recon) ** 2, axis=1)

In [12]:
def build_grid():
    common = {
        "lr": [1e-3, 3e-4],
        "weight_decay": [0.0, 1e-5],
        "batch_size": [2048, 4096, 8192],
    }

    model_spaces = {
        ContractiveAE: {
            "latent_dim": [8, 16, 32],
            "lam": [1e-4, 1e-3, 1e-2],
        },

        DenoisingAE: {
            "latent_dim": [16, 32, 64, 128, 256],
            "noise_std": [0.05, 0.1],
        },
        
        AE: {
            "latent_dim": [16, 32, 64, 128, 256],
        },

        VAE: {
            "latent_dim": [16, 32, 64, 128, 256],
            "beta": [1.0, 2.0, 5.0],
        },

        SparseAE: {
            "latent_dim": [32, 64, 128],
            "beta": [1e-3, 1e-2],
            "rho": [0.05, 0.1],
        },
    }

    grid = []
    for model_class, space in model_spaces.items():
        keys = list(common.keys()) + list(space.keys())
        values = [common[k] for k in common] + [space[k] for k in space]

        for combo in product(*values):
            config = dict(zip(keys, combo))
            config["AE_model_class"] = model_class
            grid.append(config)

    return grid

In [13]:
t_values = [
    (1.0, 0.0, 1.2, 0.4, 1.2),
    (1.0, 0.0, 1.2, 0.8, 0.8),
    (1.0, 0.4, 1.2, 0.0, 1.2),
    (1.0, 0.0, 1.2, 0.0, 0.0),
    (1.0, 0.4, 1.2, 0.4, 0.8),
    (1.0, 0.0, 1.2, 1.2, 1.2),
    (1.0, 0.0, 1.2, 0.8, 1.2),
    (1.0, 0.0, 1.2, 1.2, 0.8),
    (1.0, 0.4, 1.2, 0.0, 0.8),
    (1.0, 0.4, 1.2, 0.8, 0.8),
]

ae_param_grid = build_grid()

results = {}
for model_name in ENCODERS:
    print_big_header(f"МОДЕЛЬ: {model_name}")

    train_embeddings, test_embeddings = load_or_compute_embeddings(
        model_name,
        df_train,
        df_all
    )

    results[model_name] = {}
    for t in t_values:
        X_train = np.array(train_embeddings[0])
        X_test = np.array(test_embeddings[0])

        for i in range(1, len(t)):
            if t[i] == 0:
                continue

            X_train = np.concatenate([X_train, np.array(train_embeddings[i]) * t[i]], axis=1)
            X_test = np.concatenate([X_test, np.array(test_embeddings[i]) * t[i]], axis=1)

        y_test = df_all['label'].values.astype(int)

        for params in ae_param_grid:
            ae_scores = ae_score(X_train, X_test, params)

            auc = roc_auc_score(y_test, ae_scores) 
            ap = average_precision_score(y_test, ae_scores)
            results[model_name][(t, tuple(sorted(params.items())))] = {
                "auc": auc,
                "ap": ap,
                "params": params,
            } 
            
            param_str = ", ".join(f"{k}={v}" for k, v in params.items()) 
            print(f"Model={model_name} | t={t} | ROC-AUC={auc:.4f} | PR-AUC={ap:.4f} | {param_str}")



                                     МОДЕЛЬ: all-mpnet-base-v2                                      
[CACHE] Loading embeddings for all-mpnet-base-v2
Model=all-mpnet-base-v2 | t=(1.0, 0.0, 1.2, 0.4, 1.2) | ROC-AUC=0.9225 | PR-AUC=0.9046 | lr=0.001, weight_decay=0.0, batch_size=2048, latent_dim=8, lam=0.0001, AE_model_class=<class '__main__.ContractiveAE'>
Model=all-mpnet-base-v2 | t=(1.0, 0.0, 1.2, 0.4, 1.2) | ROC-AUC=0.9235 | PR-AUC=0.9073 | lr=0.001, weight_decay=0.0, batch_size=2048, latent_dim=8, lam=0.001, AE_model_class=<class '__main__.ContractiveAE'>
Model=all-mpnet-base-v2 | t=(1.0, 0.0, 1.2, 0.4, 1.2) | ROC-AUC=0.9231 | PR-AUC=0.9070 | lr=0.001, weight_decay=0.0, batch_size=2048, latent_dim=8, lam=0.01, AE_model_class=<class '__main__.ContractiveAE'>
Model=all-mpnet-base-v2 | t=(1.0, 0.0, 1.2, 0.4, 1.2) | ROC-AUC=0.9226 | PR-AUC=0.9054 | lr=0.001, weight_decay=0.0, batch_size=2048, latent_dim=16, lam=0.0001, AE_model_class=<class '__main__.ContractiveAE'>
Model=all-mpnet-bas

In [25]:
TOP_K = 10

for model_name, model_results in results.items():
    print_big_header(f"TOP {TOP_K} — {model_name}")

    rows = []
    for (t, params_tuple), data in model_results.items():
        params = dict(params_tuple)

        rows.append({
            "auc": data["auc"],
            "ap": data["ap"],
            "t": t,
            "AE_model_class": params["AE_model_class"].__name__,
            "lr": params["lr"],
            "weight_decay": params["weight_decay"],
            "batch_size": params["batch_size"],
            "latent_dim": params["latent_dim"],
        })

    rows_sorted = sorted(rows, key=lambda x: x["auc"], reverse=True)

    for rank, row in enumerate(rows_sorted[:TOP_K], start=1):
        print(
            f"{rank:2d}. "
            f"ROC-AUC={row['auc']:.4f} | "
            f"PR-AUC={row['ap']:.4f} | "
            f"t={row['t']} | "
            f"ld={row['latent_dim']}\t| "
            f"lr={row['lr']}\t| "
            f"wd={row['weight_decay']}\t| "
            f"bs={row['batch_size']} | "
            f"AE={row['AE_model_class']} "
        )

    print("\n")


                                     TOP 10 — all-mpnet-base-v2                                     
 1. ROC-AUC=0.9270 | PR-AUC=0.9112 | t=(1.0, 0.4, 1.2, 0.0, 1.2) | ld=64	| lr=0.001	| wd=1e-05	| bs=4096 | AE=SparseAE 
 2. ROC-AUC=0.9267 | PR-AUC=0.9091 | t=(1.0, 0.4, 1.2, 0.0, 1.2) | ld=128	| lr=0.001	| wd=0.0	| bs=2048 | AE=SparseAE 
 3. ROC-AUC=0.9266 | PR-AUC=0.9105 | t=(1.0, 0.4, 1.2, 0.0, 1.2) | ld=128	| lr=0.001	| wd=1e-05	| bs=4096 | AE=SparseAE 
 4. ROC-AUC=0.9266 | PR-AUC=0.9114 | t=(1.0, 0.0, 1.2, 0.8, 1.2) | ld=64	| lr=0.001	| wd=0.0	| bs=8192 | AE=DenoisingAE 
 5. ROC-AUC=0.9266 | PR-AUC=0.9107 | t=(1.0, 0.0, 1.2, 0.8, 1.2) | ld=32	| lr=0.001	| wd=0.0	| bs=2048 | AE=SparseAE 
 6. ROC-AUC=0.9265 | PR-AUC=0.9120 | t=(1.0, 0.0, 1.2, 0.8, 0.8) | ld=128	| lr=0.001	| wd=0.0	| bs=2048 | AE=SparseAE 
 7. ROC-AUC=0.9265 | PR-AUC=0.9125 | t=(1.0, 0.4, 1.2, 0.8, 0.8) | ld=32	| lr=0.001	| wd=1e-05	| bs=2048 | AE=SparseAE 
 8. ROC-AUC=0.9264 | PR-AUC=0.9105 | t=(1.0, 0.0, 1.2, 1.2, 

In [21]:
t_grouped = defaultdict(list)
for model_results in results.values():
    for (t, params_tuple), data in model_results.items():
        t_grouped[t].append({
            "auc": data["auc"],
            "ap": data["ap"]
        })

t_summary = []
for t, metrics in t_grouped.items():
    max_auc = max(m["auc"] for m in metrics)
    max_ap = max(m["ap"] for m in metrics)
    t_summary.append({
        "t": t,
        "max_auc": max_auc,
        "max_ap": max_ap
    })

t_summary_sorted = sorted(t_summary, key=lambda x: x["max_auc"], reverse=True)

print_big_header(f"TOP {TOP_K} по t_values (максимальный ROC-AUC)")
for rank, row in enumerate(t_summary_sorted[:TOP_K], start=1):
    print(
        f"{rank:2d}. ROC-AUC={row['max_auc']:.4f} | "
        f"PR-AUC={row['max_ap']:.4f} | t={row['t']}"
    )


                             TOP 10 по t_values (максимальный ROC-AUC)                              
 1. ROC-AUC=0.9270 | PR-AUC=0.9112 | t=(1.0, 0.4, 1.2, 0.0, 1.2)
 2. ROC-AUC=0.9266 | PR-AUC=0.9114 | t=(1.0, 0.0, 1.2, 0.8, 1.2)
 3. ROC-AUC=0.9265 | PR-AUC=0.9137 | t=(1.0, 0.0, 1.2, 0.8, 0.8)
 4. ROC-AUC=0.9265 | PR-AUC=0.9125 | t=(1.0, 0.4, 1.2, 0.8, 0.8)
 5. ROC-AUC=0.9264 | PR-AUC=0.9108 | t=(1.0, 0.0, 1.2, 1.2, 1.2)
 6. ROC-AUC=0.9263 | PR-AUC=0.9137 | t=(1.0, 0.4, 1.2, 0.4, 0.8)
 7. ROC-AUC=0.9263 | PR-AUC=0.9110 | t=(1.0, 0.0, 1.2, 0.4, 1.2)
 8. ROC-AUC=0.9262 | PR-AUC=0.9126 | t=(1.0, 0.4, 1.2, 0.0, 0.8)
 9. ROC-AUC=0.9261 | PR-AUC=0.9147 | t=(1.0, 0.0, 1.2, 1.2, 0.8)
10. ROC-AUC=0.9254 | PR-AUC=0.9135 | t=(1.0, 0.0, 1.2, 0.0, 0.0)


In [22]:
model_scores = {}
for model_results in results.values():
    for (t, params_tuple), data in model_results.items():
        params = dict(params_tuple)
        ae_name = params["AE_model_class"].__name__

        if ae_name not in model_scores:
            model_scores[ae_name] = []

        model_scores[ae_name].append(data["auc"])

model_max_scores = [
    {"AE_model_class": k, "max_auc": max(v)}
    for k, v in model_scores.items()
]

model_max_scores_sorted = sorted(model_max_scores, key=lambda x: x["max_auc"], reverse=True)

print_big_header(f"TOP {TOP_K} по AE_model_class (максимальный ROC-AUC)")
for rank, row in enumerate(model_max_scores_sorted[:TOP_K], start=1):
    print(f"{rank:2d}. AE={row['AE_model_class']} | max ROC-AUC={row['max_auc']:.4f}")


                          TOP 10 по AE_model_class (максимальный ROC-AUC)                           
 1. AE=SparseAE | max ROC-AUC=0.9270
 2. AE=DenoisingAE | max ROC-AUC=0.9266
 3. AE=AE | max ROC-AUC=0.9263
 4. AE=ContractiveAE | max ROC-AUC=0.9261
 5. AE=VAE | max ROC-AUC=0.9235


## Вывод

TODO

Autoencoder — TOP 10 по ROC-AUC:
1. ROC-AUC = 0.9268 | t = ((1, 0.0, 1.2, 1.2, 1.2), ('Autoencoder', (('batch_size', 4096), ('dropout', 0.0), ('hidden_dim', 256), ('latent_dim', 16), ('lr', 0.001), ('n_epochs', 20), ('weight_decay', 0.0)))) | params = {'latent_dim': 16, 'hidden_dim': 256, 'lr': 0.001, 'weight_decay': 0.0, 'dropout': 0.0, 'batch_size': 4096, 'n_epochs': 20}
2. ROC-AUC = 0.9265 | t = ((1, 0.4, 1.2, 0.0, 0.8), ('Autoencoder', (('batch_size', 8192), ('dropout', 0.0), ('hidden_dim', 128), ('latent_dim', 64), ('lr', 0.001), ('n_epochs', 20), ('weight_decay', 0.0)))) | params = {'latent_dim': 64, 'hidden_dim': 128, 'lr': 0.001, 'weight_decay': 0.0, 'dropout': 0.0, 'batch_size': 8192, 'n_epochs': 20}
3. ROC-AUC = 0.9264 | t = ((1, 0.0, 1.2, 1.2, 1.2), ('Autoencoder', (('batch_size', 4096), ('dropout', 0.0), ('hidden_dim', 128), ('latent_dim', 64), ('lr', 0.001), ('n_epochs', 10), ('weight_decay', 1e-05)))) | params = {'latent_dim': 64, 'hidden_dim': 128, 'lr': 0.001, 'weight_decay': 1e-05, 'dropout': 0.0, 'batch_size': 4096, 'n_epochs': 10}
4. ROC-AUC = 0.9263 | t = ((1, 0.4, 1.2, 0.0, 1.2), ('Autoencoder', (('batch_size', 4096), ('dropout', 0.0), ('hidden_dim', 256), ('latent_dim', 64), ('lr', 0.0005), ('n_epochs', 10), ('weight_decay', 1e-05)))) | params = {'latent_dim': 64, 'hidden_dim': 256, 'lr': 0.0005, 'weight_decay': 1e-05, 'dropout': 0.0, 'batch_size': 4096, 'n_epochs': 10}
5. ROC-AUC = 0.9263 | t = ((1, 0.4, 1.2, 0.4, 0.8), ('Autoencoder', (('batch_size', 4096), ('dropout', 0.0), ('hidden_dim', 256), ('latent_dim', 64), ('lr', 0.0005), ('n_epochs', 10), ('weight_decay', 0.0)))) | params = {'latent_dim': 64, 'hidden_dim': 256, 'lr': 0.0005, 'weight_decay': 0.0, 'dropout': 0.0, 'batch_size': 4096, 'n_epochs': 10}
6. ROC-AUC = 0.9263 | t = ((1, 0.0, 1.2, 0.4, 1.2), ('Autoencoder', (('batch_size', 4096), ('dropout', 0.0), ('hidden_dim', 256), ('latent_dim', 16), ('lr', 0.001), ('n_epochs', 20), ('weight_decay', 0.0)))) | params = {'latent_dim': 16, 'hidden_dim': 256, 'lr': 0.001, 'weight_decay': 0.0, 'dropout': 0.0, 'batch_size': 4096, 'n_epochs': 20}
7. ROC-AUC = 0.9263 | t = ((1, 0.4, 1.2, 0.0, 1.2), ('Autoencoder', (('batch_size', 8192), ('dropout', 0.0), ('hidden_dim', 128), ('latent_dim', 32), ('lr', 0.001), ('n_epochs', 10), ('weight_decay', 0.0)))) | params = {'latent_dim': 32, 'hidden_dim': 128, 'lr': 0.001, 'weight_decay': 0.0, 'dropout': 0.0, 'batch_size': 8192, 'n_epochs': 10}
8. ROC-AUC = 0.9261 | t = ((1, 0.4, 1.2, 0.0, 0.8), ('Autoencoder', (('batch_size', 4096), ('dropout', 0.0), ('hidden_dim', 128), ('latent_dim', 16), ('lr', 0.001), ('n_epochs', 20), ('weight_decay', 1e-05)))) | params = {'latent_dim': 16, 'hidden_dim': 128, 'lr': 0.001, 'weight_decay': 1e-05, 'dropout': 0.0, 'batch_size': 4096, 'n_epochs': 20}
9. ROC-AUC = 0.9260 | t = ((1, 0.0, 1.2, 1.2, 1.2), ('Autoencoder', (('batch_size', 4096), ('dropout', 0.0), ('hidden_dim', 256), ('latent_dim', 16), ('lr', 0.0005), ('n_epochs', 20), ('weight_decay', 0.0)))) | params = {'latent_dim': 16, 'hidden_dim': 256, 'lr': 0.0005, 'weight_decay': 0.0, 'dropout': 0.0, 'batch_size': 4096, 'n_epochs': 20}
10. ROC-AUC = 0.9260 | t = ((1, 0.4, 1.2, 0.4, 0.8), ('Autoencoder', (('batch_size', 4096), ('dropout', 0.0), ('hidden_dim', 128), ('latent_dim', 64), ('lr', 0.0005), ('n_epochs', 20), ('weight_decay', 1e-05)))) | params = {'latent_dim': 64, 'hidden_dim': 128, 'lr': 0.0005, 'weight_decay': 1e-05, 'dropout': 0.0, 'batch_size': 4096, 'n_epochs': 20}